# Lit Review Agent — Python Notebook

A faithful port of the TypeScript/Express **Lit Review Agent v2** into a single Jupyter notebook. Reuses the **same SQLite database** at `~/.litreview/data.db` so the webapp and notebook share state.

**What's inside**
1. Setup & environment
2. SQLite store (articles, reviews, parse outputs, settings)
3. PDF parsing — GROBID + OpenDataLoader hybrid + pypdf fallback
4. OpenRouter LLM client (OpenAI-compatible)
5. TEI / parser-output utilities
6. Prompts (review tasks 1/2/3, related-works, multi-paper chat)
7. Ingestion pipeline: PDF → parse → upsert
8. Per-article reviews (Task 1 metadata, Task 2 section-by-section, Task 3 related work)
9. Multi-paper context builder (citation index + cite-link style)
10. Multi-paper chat (lit-review synthesis, related-works compile, structured related-works)
11. Library helpers (search, list, delete)

**Requirements**: `pip install -r requirements-notebook.txt`. An `OPENROUTER_API_KEY` in `.env` is required for any LLM step.

## 1. Setup & environment

Loads `.env` from the project root, configures the OpenRouter key, and pins the same `~/.litreview/data.db` location used by the webapp.

In [ ]:
from __future__ import annotations

import hashlib
import io
import json
import os
import re
import sqlite3
import textwrap
import time
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

LITREVIEW_DIR = Path.home() / ".litreview"
LITREVIEW_DIR.mkdir(parents=True, exist_ok=True)
DB_PATH = LITREVIEW_DIR / "data.db"

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY", "").strip()
GROBID_URL_DEFAULT = os.getenv("GROBID_URL", "http://localhost:8070").rstrip("/")
OPENDATALOADER_HYBRID_URL_DEFAULT = os.getenv(
    "OPENDATALOADER_HYBRID_URL", "http://localhost:5002"
).rstrip("/")

print(f"DB path:               {DB_PATH}")
print(f"OpenRouter key set:    {bool(OPENROUTER_API_KEY)}")
print(f"GROBID URL (default):  {GROBID_URL_DEFAULT}")
print(f"ODL hybrid (default):  {OPENDATALOADER_HYBRID_URL_DEFAULT}")

## 2. SQLite store

Mirrors the `app/db.ts` schema 1:1 (articles, reviews, settings, article_parse_outputs) including foreign-key cascades. Safe to run alongside the webapp — both processes share the same DB.

In [ ]:
_db: sqlite3.Connection | None = None


def get_db() -> sqlite3.Connection:
    """Return a shared SQLite connection with the same schema as app/db.ts."""
    global _db
    if _db is not None:
        return _db
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON")
    conn.executescript(
        """
        CREATE TABLE IF NOT EXISTS articles (
            id TEXT PRIMARY KEY,
            title TEXT,
            authors TEXT,
            abstract TEXT,
            pdf_path TEXT,
            xml TEXT,
            parsed_at TEXT,
            model_used TEXT
        );
        CREATE TABLE IF NOT EXISTS reviews (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            article_id TEXT REFERENCES articles(id) ON DELETE CASCADE,
            task INTEGER NOT NULL,
            model TEXT NOT NULL,
            result TEXT NOT NULL,
            created_at TEXT DEFAULT (datetime('now')),
            UNIQUE(article_id, task, model)
        );
        CREATE TABLE IF NOT EXISTS settings (
            key TEXT PRIMARY KEY,
            value TEXT
        );
        CREATE TABLE IF NOT EXISTS article_parse_outputs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            article_id TEXT NOT NULL REFERENCES articles(id) ON DELETE CASCADE,
            parser_engine TEXT NOT NULL,
            parser_model TEXT,
            output_format TEXT NOT NULL,
            payload_json TEXT,
            normalized_text TEXT,
            is_primary INTEGER NOT NULL DEFAULT 0,
            created_at TEXT DEFAULT (datetime('now'))
        );
        CREATE INDEX IF NOT EXISTS idx_parse_outputs_article
            ON article_parse_outputs(article_id, created_at DESC);
        """
    )

    def cols(table: str) -> set[str]:
        return {row["name"] for row in conn.execute(f"PRAGMA table_info({table})")}

    article_cols = cols("articles")
    for col, ddl in [
        ("year", "ALTER TABLE articles ADD COLUMN year INTEGER"),
        ("venue_type", "ALTER TABLE articles ADD COLUMN venue_type TEXT"),
        ("venue_name", "ALTER TABLE articles ADD COLUMN venue_name TEXT"),
        ("links_json", "ALTER TABLE articles ADD COLUMN links_json TEXT"),
        ("folder", "ALTER TABLE articles ADD COLUMN folder TEXT"),
        ("parser_engine", "ALTER TABLE articles ADD COLUMN parser_engine TEXT"),
    ]:
        if col not in article_cols:
            conn.execute(ddl)

    review_cols = cols("reviews")
    if "review_depth" not in review_cols:
        conn.executescript(
            """
            CREATE TABLE IF NOT EXISTS reviews_migrated (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                article_id TEXT NOT NULL REFERENCES articles(id) ON DELETE CASCADE,
                task INTEGER NOT NULL,
                model TEXT NOT NULL,
                review_depth TEXT NOT NULL DEFAULT '',
                result TEXT NOT NULL,
                created_at TEXT DEFAULT (datetime('now')),
                UNIQUE(article_id, task, model, review_depth)
            );
            INSERT INTO reviews_migrated
                (article_id, task, model, review_depth, result, created_at)
            SELECT article_id, task, model,
                   CASE WHEN task = 2 THEN 'detailed' ELSE '' END,
                   result, created_at
            FROM reviews;
            DROP TABLE reviews;
            ALTER TABLE reviews_migrated RENAME TO reviews;
            """
        )

    conn.execute(
        """
        INSERT INTO settings (key, value)
        SELECT 'pdf_parser_default_engine', 'opendataloader'
        WHERE NOT EXISTS (SELECT 1 FROM settings WHERE key = 'pdf_parser_default_engine')
        """
    )
    conn.commit()
    _db = conn
    return conn


def get_setting(key: str) -> str | None:
    row = get_db().execute("SELECT value FROM settings WHERE key = ?", (key,)).fetchone()
    return row["value"] if row else None


def set_setting(key: str, value: str) -> None:
    get_db().execute(
        "INSERT INTO settings (key, value) VALUES (?, ?) "
        "ON CONFLICT(key) DO UPDATE SET value = excluded.value",
        (key, value),
    )
    get_db().commit()


get_db()
print("DB ready at", DB_PATH)

In [ ]:
def folder_from_pdf_path(pdf_path: str | None) -> str | None:
    if not pdf_path:
        return None
    n = pdf_path.replace("\\", "/")
    i = n.rfind("/")
    if i <= 0:
        return None
    return n[:i] or None


def upsert_article(article: dict[str, Any]) -> None:
    """Insert/merge an article row, preserving non-null existing values (matches app/db.ts COALESCE)."""
    cols = [
        "id", "title", "authors", "abstract", "pdf_path", "xml", "parsed_at",
        "model_used", "year", "venue_type", "venue_name", "links_json",
        "folder", "parser_engine",
    ]
    values = [article.get(c) for c in cols]
    placeholders = ", ".join("?" for _ in cols)
    set_clause = ", ".join(f"{c} = COALESCE(excluded.{c}, {c})" for c in cols if c != "id")
    sql = (
        f"INSERT INTO articles ({', '.join(cols)}) VALUES ({placeholders}) "
        f"ON CONFLICT(id) DO UPDATE SET {set_clause}"
    )
    get_db().execute(sql, values)
    get_db().commit()


def get_article(article_id: str) -> dict[str, Any] | None:
    row = get_db().execute(
        "SELECT id, title, authors, abstract, pdf_path, xml, parsed_at, model_used, "
        "year, venue_type, venue_name, links_json, folder, parser_engine "
        "FROM articles WHERE id = ?",
        (article_id,),
    ).fetchone()
    return dict(row) if row else None


def get_articles(
    *,
    search: str | None = None,
    year_min: int | None = None,
    year_max: int | None = None,
    venue_type: str | None = None,
    folder: str | None = None,
    sort: str = "parsed_at",
    order: str = "desc",
    include_xml: bool = True,
) -> list[dict[str, Any]]:
    select_cols = (
        "id, title, authors, abstract, pdf_path, xml, parsed_at, model_used, "
        "year, venue_type, venue_name, links_json, folder, parser_engine"
        if include_xml
        else "id, title, authors, abstract, pdf_path, NULL AS xml, parsed_at, model_used, "
             "year, venue_type, venue_name, links_json, folder, parser_engine"
    )
    sql = f"SELECT {select_cols} FROM articles WHERE 1=1"
    params: list[Any] = []
    if search:
        sql += (
            " AND (title LIKE ? OR abstract LIKE ? OR authors LIKE ? "
            "OR IFNULL(venue_name,'') LIKE ? OR IFNULL(links_json,'') LIKE ? "
            "OR IFNULL(folder,'') LIKE ?)"
        )
        term = f"%{search}%"
        params += [term] * 6
    if year_min is not None:
        sql += " AND year IS NOT NULL AND year >= ?"
        params.append(year_min)
    if year_max is not None:
        sql += " AND year IS NOT NULL AND year <= ?"
        params.append(year_max)
    if venue_type and venue_type != "all":
        sql += " AND IFNULL(venue_type,'') = ?"
        params.append(venue_type.strip())
    if folder == "__root__":
        sql += " AND (folder IS NULL OR TRIM(IFNULL(folder,'')) = '')"
    elif folder:
        sql += " AND folder = ?"
        params.append(folder.strip())
    order_kw = "ASC" if order.lower() == "asc" else "DESC"
    if sort == "folder":
        sql += f" ORDER BY IFNULL(folder,'') {order_kw}, parsed_at DESC"
    else:
        sort_col = "title" if sort == "title" else "parsed_at"
        sql += f" ORDER BY {sort_col} {order_kw}"
    return [dict(r) for r in get_db().execute(sql, params)]


def delete_article(article_id: str) -> None:
    get_db().execute("DELETE FROM articles WHERE id = ?", (article_id,))
    get_db().commit()


def delete_all_articles() -> int:
    cur = get_db().execute("DELETE FROM articles")
    get_db().commit()
    return cur.rowcount or 0


def get_review(article_id: str, task: int, model: str, review_depth: str = "") -> dict[str, Any] | None:
    row = get_db().execute(
        "SELECT id, article_id, task, model, review_depth, result, created_at "
        "FROM reviews WHERE article_id = ? AND task = ? AND model = ? AND review_depth = ?",
        (article_id, task, model, review_depth),
    ).fetchone()
    return dict(row) if row else None


def get_reviews(article_id: str) -> list[dict[str, Any]]:
    return [
        dict(r)
        for r in get_db().execute(
            "SELECT id, article_id, task, model, review_depth, result, created_at "
            "FROM reviews WHERE article_id = ? ORDER BY created_at DESC",
            (article_id,),
        )
    ]


def upsert_review(article_id: str, task: int, model: str, result: str, review_depth: str = "") -> None:
    get_db().execute(
        """
        INSERT INTO reviews (article_id, task, model, review_depth, result)
        VALUES (?, ?, ?, ?, ?)
        ON CONFLICT(article_id, task, model, review_depth)
        DO UPDATE SET result = excluded.result, created_at = datetime('now')
        """,
        (article_id, task, model, review_depth, result),
    )
    get_db().commit()


def insert_parse_output(
    *,
    article_id: str,
    parser_engine: str,
    parser_model: str | None,
    output_format: str,
    payload_json: str | None = None,
    normalized_text: str | None = None,
    is_primary: bool = False,
) -> int:
    db = get_db()
    if is_primary:
        db.execute(
            "UPDATE article_parse_outputs SET is_primary = 0 WHERE article_id = ?",
            (article_id,),
        )
    cur = db.execute(
        "INSERT INTO article_parse_outputs "
        "(article_id, parser_engine, parser_model, output_format, payload_json, normalized_text, is_primary) "
        "VALUES (?, ?, ?, ?, ?, ?, ?)",
        (article_id, parser_engine, parser_model, output_format, payload_json,
         normalized_text, 1 if is_primary else 0),
    )
    db.commit()
    return int(cur.lastrowid)


def get_latest_parse_output(article_id: str) -> dict[str, Any] | None:
    row = get_db().execute(
        "SELECT id, article_id, parser_engine, parser_model, output_format, "
        "payload_json, normalized_text, is_primary, created_at "
        "FROM article_parse_outputs WHERE article_id = ? "
        "ORDER BY is_primary DESC, created_at DESC LIMIT 1",
        (article_id,),
    ).fetchone()
    return dict(row) if row else None


def article_count() -> int:
    return int(get_db().execute("SELECT COUNT(*) AS n FROM articles").fetchone()["n"])


def review_count() -> int:
    return int(get_db().execute("SELECT COUNT(*) AS n FROM reviews").fetchone()["n"])


print(f"Articles: {article_count()}  Reviews: {review_count()}")

## 3. PDF parsing

Three engines, in priority order:

1. **GROBID** — TEI XML. Run `docker run --rm -p 8070:8070 lfoppiano/grobid:0.8.0`.
2. **OpenDataLoader hybrid** — JSON + Markdown. Run `pip install "opendataloader-pdf[hybrid]" && opendataloader-pdf-hybrid --port 5002`.
3. **pypdf fallback** — pure Python plain-text extraction (always available).

Each parser returns a normalized record stored via `insert_parse_output`. GROBID's TEI also lands in `articles.xml` because the original webapp keeps it there for downstream review tasks.

In [ ]:
def _grobid_url(override: str | None = None) -> str:
    return (override or get_setting("grobid_url") or GROBID_URL_DEFAULT).rstrip("/")


def grobid_alive(base_url: str | None = None) -> bool:
    try:
        r = requests.get(f"{_grobid_url(base_url)}/api/isalive", timeout=3)
        return r.ok
    except Exception:
        return False


def parse_pdf_grobid(pdf_bytes: bytes, filename: str, base_url: str | None = None) -> str:
    """Send a PDF to GROBID's processFulltextDocument; returns TEI XML as str."""
    url = f"{_grobid_url(base_url)}/api/processFulltextDocument"
    files = {"input": (filename, pdf_bytes, "application/pdf")}
    data = {"consolidateHeader": "1", "consolidateCitations": "0"}
    r = requests.post(url, files=files, data=data, timeout=300)
    if not r.ok:
        raise RuntimeError(f"GROBID error ({r.status_code}): {r.text[:500]}")
    return r.text


def _opendataloader_url(override: str | None = None) -> str:
    return (
        override
        or get_setting("opendataloader_hybrid_url")
        or OPENDATALOADER_HYBRID_URL_DEFAULT
    ).rstrip("/")


def opendataloader_alive(base_url: str | None = None) -> bool:
    try:
        r = requests.get(f"{_opendataloader_url(base_url)}/health", timeout=3)
        return r.ok
    except Exception:
        return False


def parse_pdf_opendataloader(
    pdf_bytes: bytes, filename: str, base_url: str | None = None
) -> dict[str, Any]:
    """Hybrid OpenDataLoader server — returns {'json': dict, 'markdown': str} when reachable."""
    url = f"{_opendataloader_url(base_url)}/parse"
    files = {"file": (filename, pdf_bytes, "application/pdf")}
    r = requests.post(url, files=files, timeout=600)
    if not r.ok:
        raise RuntimeError(
            f"OpenDataLoader hybrid error ({r.status_code}): {r.text[:500]}"
        )
    body = r.json()
    md = body.get("markdown") or body.get("md") or ""
    return {"json": body, "markdown": md}


def parse_pdf_pypdf(pdf_bytes: bytes) -> str:
    """Pure-Python fallback: extract concatenated page text via pypdf."""
    from pypdf import PdfReader

    reader = PdfReader(io.BytesIO(pdf_bytes))
    pages: list[str] = []
    for i, page in enumerate(reader.pages, start=1):
        try:
            pages.append(f"\n\n--- Page {i} ---\n\n{page.extract_text() or ''}")
        except Exception as e:
            pages.append(f"\n\n--- Page {i} (extract failed: {e}) ---\n")
    return "".join(pages).strip()


@dataclass
class ParseResult:
    engine: str            # "grobid" | "opendataloader" | "pypdf"
    output_format: str     # "tei_xml" | "json" | "markdown" | "text"
    payload_json: str | None = None
    xml: str | None = None
    normalized_text: str | None = None


def parse_pdf(
    pdf_bytes: bytes,
    filename: str,
    *,
    prefer: str | None = None,
    grobid_base: str | None = None,
    odl_base: str | None = None,
) -> ParseResult:
    """Try the chosen parser, falling back through the chain to pypdf."""
    order = []
    if prefer:
        order.append(prefer)
    for cand in ("grobid", "opendataloader", "pypdf"):
        if cand not in order:
            order.append(cand)

    last_err: Exception | None = None
    for engine in order:
        try:
            if engine == "grobid" and grobid_alive(grobid_base):
                tei = parse_pdf_grobid(pdf_bytes, filename, grobid_base)
                return ParseResult(
                    engine="grobid",
                    output_format="tei_xml",
                    xml=tei,
                    normalized_text=strip_tei_tags(tei),
                )
            if engine == "opendataloader" and opendataloader_alive(odl_base):
                out = parse_pdf_opendataloader(pdf_bytes, filename, odl_base)
                return ParseResult(
                    engine="opendataloader",
                    output_format="markdown" if out["markdown"] else "json",
                    payload_json=json.dumps(out["json"], ensure_ascii=False),
                    normalized_text=out["markdown"] or json.dumps(out["json"])[:200_000],
                )
            if engine == "pypdf":
                txt = parse_pdf_pypdf(pdf_bytes)
                return ParseResult(
                    engine="pypdf",
                    output_format="text",
                    normalized_text=txt,
                )
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"All parsers failed. Last error: {last_err}")


print("GROBID alive:        ", grobid_alive())
print("OpenDataLoader alive:", opendataloader_alive())

## 4. OpenRouter LLM client

Drop-in replacement for `app/lib/openrouter.ts`. OpenRouter is OpenAI-compatible, so we use the `openai` SDK pointed at `https://openrouter.ai/api/v1`. Streaming is wrapped to yield text deltas the same way the webapp's SSE handler does.

In [ ]:
try:
    from openai import OpenAI
except ImportError as e:
    raise ImportError("Install deps: pip install -r requirements-notebook.txt") from e


def openrouter_client(api_key: str | None = None) -> OpenAI:
    key = (
        api_key
        or get_setting("openrouter_api_key")
        or OPENROUTER_API_KEY
    )
    if not key:
        raise RuntimeError(
            "OPENROUTER_API_KEY missing. Add it to .env or call set_setting('openrouter_api_key', ...)."
        )
    return OpenAI(
        api_key=key,
        base_url="https://openrouter.ai/api/v1",
        default_headers={
            "HTTP-Referer": "https://github.com/preethamam/Papers-Articles-Literature-Review-Agent",
            "X-Title": "Lit Review Agent (Notebook)",
        },
    )


def chat_complete(
    messages: list[dict[str, str]],
    *,
    model: str | None = None,
    stream: bool = True,
    print_stream: bool = True,
) -> str:
    """Call OpenRouter and return full text. When stream=True and print_stream=True,
    deltas are echoed live (mirrors the SSE behavior of /api/reviews and /api/chat)."""
    chosen = model or get_setting("default_model") or "openrouter/auto"
    client = openrouter_client()
    if not stream:
        resp = client.chat.completions.create(model=chosen, messages=messages)
        return resp.choices[0].message.content or ""

    chunks: list[str] = []
    resp = client.chat.completions.create(model=chosen, messages=messages, stream=True)
    for chunk in resp:
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        piece = (getattr(delta, "content", None) or "") + (
            getattr(delta, "reasoning", None) or ""
        )
        if piece:
            chunks.append(piece)
            if print_stream:
                print(piece, end="", flush=True)
    if print_stream:
        print()
    return "".join(chunks)


def list_openrouter_models(query: str | None = None, limit: int = 30) -> list[dict[str, Any]]:
    r = requests.get(
        "https://openrouter.ai/api/v1/models",
        headers={"Authorization": f"Bearer {OPENROUTER_API_KEY}"} if OPENROUTER_API_KEY else {},
        timeout=20,
    )
    r.raise_for_status()
    models = r.json().get("data", [])
    if query:
        q = query.lower()
        models = [m for m in models if q in m.get("id", "").lower() or q in (m.get("name", "") or "").lower()]
    return models[:limit]


print("OpenRouter client ready" if OPENROUTER_API_KEY else "WARNING: no OPENROUTER_API_KEY set")

## 5. TEI / parser-output utilities

Ports of `app/lib/teiSectionExtract.ts`. Used to slice "Related Work" and "Conclusion" out of GROBID TEI for focused multi-paper context.

In [ ]:
_TAG_RE = re.compile(r"<[^>]+>")
_WS_RE = re.compile(r"\s+")


def strip_tei_tags(xml: str) -> str:
    return _WS_RE.sub(" ", _TAG_RE.sub(" ", xml or "")).strip()


def _is_related_work_head(label: str) -> bool:
    t = strip_tei_tags(label).lower()
    return bool(
        re.search(r"related\s+work", t)
        or re.search(r"prior\s+work", t)
        or re.match(r"^background\s*$", t)
        or re.search(r"background\s+and\s+related", t)
        or re.search(r"literature\s+review", t)
        or re.search(r"literature\s+survey", t)
    )


def _is_conclusion_head(label: str) -> bool:
    t = strip_tei_tags(label).lower().strip()
    return bool(
        t.startswith("conclusion")
        or t.startswith("conclusions")
        or re.search(r"discussion\s+and\s+conclusion", t)
        or t == "summary"
        or re.search(r"concluding\s+remarks", t)
    )


def _extract_after_head(xml: str, predicate, max_slice: int) -> str | None:
    if not (xml or "").strip():
        return None
    head_re = re.compile(r"<head[^>]*>([\s\S]*?)</head>", re.IGNORECASE)
    for m in head_re.finditer(xml):
        if not predicate(m.group(1)):
            continue
        after = xml[m.end():]
        next_head = re.search(r"<head[^>]", after, re.IGNORECASE)
        slice_ = after[: next_head.start()] if next_head else after[:max_slice]
        text = strip_tei_tags(slice_)
        if len(text) > 40:
            return text[:50_000]
    return None


def extract_related_work_from_tei(xml: str) -> str | None:
    return _extract_after_head(xml, _is_related_work_head, 20_000)


def extract_conclusion_from_tei(xml: str) -> str | None:
    return _extract_after_head(xml, _is_conclusion_head, 15_000)


def tei_metadata(xml: str) -> dict[str, Any]:
    """Cheap regex-based pull of common header fields from GROBID TEI."""
    if not xml:
        return {}
    out: dict[str, Any] = {}
    title = re.search(
        r"<title[^>]*type=\"main\"[^>]*>(.*?)</title>", xml, re.IGNORECASE | re.DOTALL
    ) or re.search(r"<title[^>]*>(.*?)</title>", xml, re.IGNORECASE | re.DOTALL)
    if title:
        out["title"] = strip_tei_tags(title.group(1))
    abstract = re.search(r"<abstract[^>]*>([\s\S]*?)</abstract>", xml, re.IGNORECASE)
    if abstract:
        out["abstract"] = strip_tei_tags(abstract.group(1))
    authors = []
    for am in re.finditer(r"<author[^>]*>([\s\S]*?)</author>", xml, re.IGNORECASE):
        nm = re.search(
            r"<persName[^>]*>([\s\S]*?)</persName>", am.group(1), re.IGNORECASE
        )
        if nm:
            authors.append(strip_tei_tags(nm.group(1)))
    if authors:
        out["authors"] = "; ".join(authors)
    year = re.search(r'when="(\d{4})', xml)
    if year:
        out["year"] = int(year.group(1))
    return out


print("TEI utilities loaded.")

## 6. Prompts

Direct ports of the most-used strings in `app/lib/prompts.ts` — the academic-style block, paper-review system prompt (tasks 1/2/3), Task-2 depth instructions (one_line / five_line / detailed), the multi-paper synthesis system prompt, and the related-works compile/structured prompts.

In [ ]:
ACADEMIC_STYLE_BLOCK = """**Writing style**
- Use precise, neutral academic English appropriate to STEM and social-science papers.
- Prefer short sentences, commas, or semicolons; do not overuse em-dashes for rhetorical effect.
- Avoid hype ("groundbreaking", "revolutionary"), vague transitions ("Moreover", "It is worth noting" without substance), and filler.
- State claims at a level supported by the supplied text only.""".strip()


TASK2_DEPTH_INSTRUCTIONS: dict[str, str] = {
    "one_line":
        "**Depth: one line per section.** For each section, output exactly one clear sentence "
        "capturing the main point. No bullet lists unless the section is trivial.",
    "five_line":
        "**Depth: about five lines per section.** For each section, write a short paragraph "
        "(roughly 4–6 sentences) covering purpose, method, key result, and limitation if visible.",
    "detailed": (
        "**Depth: detailed — strictly section-by-section.**\n\n"
        "**Hard requirements**\n"
        "- Emit one `## <section number>. <Section name>` heading **for every `<section>` in the provided XML**, "
        "in the order they appear. Do **not** merge, skip, or rename sections; copy the section `name` verbatim.\n"
        "- Under each heading, write **a thorough multi-paragraph treatment** (2–4 paragraphs, ~150–300 words) "
        "grounded only in that section's text.\n"
        "- For each section, cover in order: (1) purpose, (2) key claims/methods/definitions, (3) data/equations/evidence, "
        "(4) results/findings, (5) limitations or open questions.\n"
        "- Nest subsections as `### <sub-number>. <sub-name>` in document order.\n"
        "- If a section is empty, say so explicitly rather than skipping it.\n"
        "- End with `## Overall synthesis` (4–6 bullets tying sections together).\n\n"
        "**Style**\n"
        "- Prefer dense academic paragraphs over bullet lists inside each section.\n"
        "- Quote short distinctive phrases sparingly when terminology matters; otherwise paraphrase.\n"
        "- Ground every statement in that section's XML text."
    ),
}


PAPER_REVIEW_SYSTEM = f"""You are a research paper review agent that processes XML metadata of academic papers. The XML document contains structured information about a paper, including title, authors, abstract, sections, and links. Your input consists of:
1. An XML document (with tags like `<title>`, `<authors>`, `<abstract>`, `<sections>` containing `<section>` elements with a `name` attribute, and optionally `<links>` or links within sections).
2. A task option number: 1, 2, or 3.

Perform the selected task as follows:

### **Option 1: Extract metadata and open source code and datasets links**
- Extract:
  - **Title**: From `<title>`.
  - **Authors**: List from `<authors>/<author>`.
  - **Abstract**: Full text from `<abstract>`.
  - **Links**:
    - Scan all `<section>` contents and any dedicated `<links>` section for URLs (strings starting with `http://` or `https://`).
    - For each URL found, capture: `url`, `description`, `type` (one of `code`, `dataset`, `demo`, `video`, `paper`, `other`).
- Output a JSON object with keys: `"title"`, `"authors"`, `"abstract"`, `"links"`.

### **Option 2: Section-by-section summary**
- Iterate over **every** `<section>` in `<sections>`, in document order.
- For each section emit `## <n>. <Section name>` and follow the depth instructions provided.
- Render nested subsections as `### <n.m>. <Sub-name>`.
- If a section contains no substantive prose, state that explicitly rather than dropping it.
- Output is a single Markdown document.

### **Option 3: Related work synthesizing agent**
- Identify the section discussing prior work (Related Work / Prior Work / Background).
- Summarize key prior works mentioned: authors, year, approach, relevance.
- Synthesize emerging themes, gaps in prior work, and how this paper addresses them.
- If the related work section is sparse, infer from introduction/conclusion.
- Output a cohesive paragraph highlighting the evolution of ideas and the paper's contribution relative to prior art.

**General Instructions**:
- Handle missing tags gracefully; use closest matches.
- Ground every statement only in the provided XML content.
- Do not cite, mention, or infer any external paper not in the supplied XML context.
- If evidence is missing, explicitly state "insufficient evidence in provided context".

---

{ACADEMIC_STYLE_BLOCK}"""


def get_literature_synthesis_system(detail_level: int = 0) -> str:
    base = ACADEMIC_STYLE_BLOCK
    if detail_level >= 2:
        return f"""{base}

You are helping a researcher produce a **literature review synthesis** across multiple papers provided in context.

**Goals**
- Produce a deeply structured synthesis with clear thematic sections, sub-themes, and explicit cross-paper comparisons.
- Treat methodological assumptions, datasets/evaluation settings, and points of disagreement.
- End each theme with implications and open gaps.

**Output**
- Use section headings and substantial narrative depth.
- At detail level 3, include a short "Research agenda" subsection grounded in evidence.

**Citations**
- Use the Markdown citation links exactly as specified in the context block (`[n](cite:INTERNAL_ID)`) for any specific claim tied to a document.
- Do not cite or name papers that are not in the provided context."""
    if detail_level == 1:
        return f"""{base}

You are helping a researcher produce a **literature review synthesis** across multiple papers provided in context.

**Goals**
- Build a comprehensive narrative with explicit thematic sections (methods, datasets, findings, limitations, open gaps).
- Compare and contrast papers within each theme, including disagreements and methodological trade-offs when present.

**Citations**
- Use `[n](cite:INTERNAL_ID)` exactly as in the context block.
- Do not cite or name papers not in context.

If evidence is insufficient for a point, say so briefly rather than speculating."""
    return f"""{base}

You are helping a researcher produce a **literature review synthesis** across multiple papers provided in context (not paper-by-paper bullets unless asked).

**Goals**
- Integrate themes, contrasts, and gaps across the set.
- Note methodological tendencies, datasets, and conflicting findings where evidence exists in the text.
- Use an integrated narrative (multiple paragraphs).

**Citations**
- Use `[n](cite:INTERNAL_ID)` exactly as in the context block.
- Do not cite papers outside the provided context."""


_RELATED_WORK_BAR = """**Research-grade bar (non-negotiable)**
- Write for an expert reader: comparative, skeptical, and precise. Avoid generic filler.
- Organize by **themes / research threads**. Within each theme, **compare across papers**.
- Separate **supported** claims (with citations) from **tentative** interpretations; if excerpts do not support a stronger claim, say "not evidenced in the provided text".
- Surface **tensions and disagreements** when papers conflict; do not flatten contradictions.
- End with **limitations, gaps, and open questions** grounded in what the texts actually say.""".strip()


def get_related_work_compile_system(detail_level: int = 0) -> str:
    base = ACADEMIC_STYLE_BLOCK
    if detail_level >= 2:
        agenda = "## Research agenda\n- 4–6 concrete next steps tied to gaps you identified above." if detail_level >= 3 else ""
        return f"""{base}

You are writing a **Related Works** section. The user selected 2–25 papers; produce **thematic synthesis and comparison**, not a catalog of abstracts.

{_RELATED_WORK_BAR}

**Output structure**
## Related Works
### Landscape and scope
### Thematic synthesis
### Limitations, conflicts, and gaps
## Synthesis takeaways
- {('8–12' if detail_level >= 3 else '6–10')} bullets

{agenda}

**Citations**
- Use `[n](cite:INTERNAL_ID)` for every substantive paper-specific claim.
- Do not cite papers not in the context."""
    if detail_level == 1:
        return f"""{base}

You are writing a **Related Works** section across 2–25 papers as integrated synthesis.

{_RELATED_WORK_BAR}

## Related Works
### Thematic overview
### Limitations and gaps
## Synthesis takeaways
- 5–8 bullets

**Citations**
- Use `[n](cite:INTERNAL_ID)` for specific claims tied to a paper."""
    return f"""{base}

You are writing a concise but research-grade **Related Works** across 2–25 papers.

{_RELATED_WORK_BAR}

## Related Works
- Thematic narrative (2–4 themes), then **Limitations & gaps**.
## Synthesis takeaways
- 4–6 bullets

**Citations**
- Use `[n](cite:INTERNAL_ID)` for substantive paper-specific claims."""


def get_related_work_structured_system(detail_level: int = 0) -> str:
    if detail_level >= 3:
        depth = "Use **longer** bullets and paragraphs where evidence supports it; keep claims grounded."
    elif detail_level >= 2:
        depth = "Use substantive bullets (not single words) for strengths/weaknesses/critic."
    else:
        depth = "Stay compact but complete every required bullet for each paper."
    return f"""{ACADEMIC_STYLE_BLOCK}

You produce a **structured related-works report** across 2–25 papers in context. {depth}

{_RELATED_WORK_BAR}

## Per-paper cards
For each paper in citation-index order (`Paper [1]`, `Paper [2]`, ...), emit:

### [n] <short recognizable title>
- **One-line:** one sentence on contribution.
- **Strengths:** 2–4 bullets.
- **Weaknesses:** 2–4 bullets.
- **Critic:** 2–5 sentences of independent assessment.

Every factual statement ends with `[n](cite:INTERNAL_ID)`.

## Section-by-section synthesis
### Problem framing
### Methods
### Datasets / metrics
### Findings
### Limitations & open questions

Dense comparative prose; frequent `[n](cite:INTERNAL_ID)` citations.

**Important**: Do not emit a `## BibTeX` section yourself.

**Citations**
- Use `[n](cite:INTERNAL_ID)` as in the context block.
- Do not cite papers not in context."""


print("Prompts loaded.")

## 7. Ingestion pipeline

`ingest_pdf(path)` reads a PDF, runs the parser chain, derives a stable article ID (sha-1 of bytes — same scheme used by the webapp's hash util), upserts the article, records the parse output, and (when GROBID is used) populates `articles.xml` plus extracted metadata.

Pass `prefer="grobid"|"opendataloader"|"pypdf"` to override engine selection.

In [ ]:
def article_id_for(pdf_bytes: bytes) -> str:
    """Stable id = sha1 of the PDF bytes (matches the webapp's content-hash convention)."""
    return hashlib.sha1(pdf_bytes).hexdigest()


def ingest_pdf(
    pdf_path: str | Path,
    *,
    prefer: str | None = None,
    folder: str | None = None,
    grobid_base: str | None = None,
    odl_base: str | None = None,
) -> dict[str, Any]:
    """End-to-end: read PDF → parse → upsert article → record parse output."""
    p = Path(pdf_path).expanduser().resolve()
    pdf_bytes = p.read_bytes()
    aid = article_id_for(pdf_bytes)

    result = parse_pdf(
        pdf_bytes,
        p.name,
        prefer=prefer,
        grobid_base=grobid_base,
        odl_base=odl_base,
    )

    meta: dict[str, Any] = {}
    if result.engine == "grobid" and result.xml:
        meta = tei_metadata(result.xml)

    upsert_article({
        "id": aid,
        "title": meta.get("title"),
        "authors": meta.get("authors"),
        "abstract": meta.get("abstract"),
        "year": meta.get("year"),
        "pdf_path": str(p),
        "xml": result.xml,
        "parsed_at": datetime.now(timezone.utc).isoformat(),
        "model_used": None,
        "folder": folder or folder_from_pdf_path(str(p)),
        "parser_engine": result.engine,
    })
    insert_parse_output(
        article_id=aid,
        parser_engine=result.engine,
        parser_model=None,
        output_format=result.output_format,
        payload_json=result.payload_json,
        normalized_text=result.normalized_text,
        is_primary=True,
    )

    summary = {
        "id": aid,
        "engine": result.engine,
        "format": result.output_format,
        "title": meta.get("title"),
        "authors": meta.get("authors"),
        "year": meta.get("year"),
        "pdf_path": str(p),
    }
    return summary


def ingest_directory(
    directory: str | Path,
    *,
    prefer: str | None = None,
    pattern: str = "*.pdf",
    grobid_base: str | None = None,
    odl_base: str | None = None,
) -> list[dict[str, Any]]:
    """Recursively ingest all PDFs under directory."""
    root = Path(directory).expanduser().resolve()
    out: list[dict[str, Any]] = []
    pdfs = sorted(root.rglob(pattern))
    for i, p in enumerate(pdfs, 1):
        print(f"[{i}/{len(pdfs)}] {p.relative_to(root)} ... ", end="", flush=True)
        try:
            s = ingest_pdf(p, prefer=prefer, grobid_base=grobid_base, odl_base=odl_base)
            print(f"OK [{s['engine']}] -> {s['id'][:10]}…")
            out.append(s)
        except Exception as e:
            print(f"FAILED: {e}")
            out.append({"pdf_path": str(p), "error": str(e)})
    return out


print("Ingestion ready. Try: ingest_pdf('path/to/paper.pdf')")

## 8. Per-article reviews (Tasks 1, 2, 3)

Mirror of `app/routes/reviews.ts`. Each call:

- prefers TEI XML if present, else falls back to the latest parse output (Markdown/JSON);
- truncates to 200,000 chars;
- caches results in `reviews` table keyed on `(article_id, task, model, review_depth)`;
- streams the model output live.

`review_article(article_id, task=2, model=..., depth="detailed")`

In [ ]:
_TASK_LABELS = {
    1: "Extract metadata and links (Option 1)",
    2: "Section-by-section summary (Option 2)",
    3: "Related work synthesis (Option 3)",
}
_TASK2_DEPTHS = {"one_line", "five_line", "detailed"}


def review_article(
    article_id: str,
    *,
    task: int = 2,
    model: str | None = None,
    depth: str = "detailed",
    use_cache: bool = True,
    print_stream: bool = True,
) -> dict[str, Any]:
    """Run one of the three review tasks against a stored article."""
    if task not in (1, 2, 3):
        raise ValueError("task must be 1, 2, or 3")

    article = get_article(article_id)
    if not article:
        raise KeyError(f"Article {article_id!r} not found")

    review_depth = depth if (task == 2 and depth in _TASK2_DEPTHS) else ""
    effective_depth = (depth if depth in _TASK2_DEPTHS else "detailed") if task == 2 else ""

    tei = (article.get("xml") or "").strip()
    if tei:
        document_body = tei[:200_000]
        document_kind = "tei"
    else:
        latest = get_latest_parse_output(article_id)
        payload = (latest or {}).get("payload_json") or ""
        normalized = (latest or {}).get("normalized_text") or ""
        body = (payload or normalized).strip()
        if not body:
            raise RuntimeError("Article has no parseable content (no TEI XML or parser output)")
        document_body = body[:200_000]
        fmt = ((latest or {}).get("output_format") or "").lower()
        document_kind = "json" if fmt == "json" else "markdown"

    chosen_model = (
        model
        or get_setting(f"default_model_task{task}")
        or get_setting("default_model")
        or "openrouter/auto"
    )

    if use_cache:
        cached = get_review(article_id, task, chosen_model, effective_depth)
        if cached:
            if print_stream:
                print("[cached]")
                print(cached["result"])
            return {
                "cached": True,
                "result": cached["result"],
                "model": chosen_model,
                "review_depth": effective_depth,
            }

    label = _TASK_LABELS[task]
    user_msg = f"Task option: {task}\n\n"
    if document_kind != "tei":
        user_msg += (
            "The document below was produced by a PDF parser (Markdown or JSON layout), "
            "not TEI XML. Apply the same task goals; treat headings and blocks as section "
            "boundaries where applicable.\n\n"
        )
    if document_kind == "tei":
        user_msg += f"Please process the uploaded TEI XML using option {task}: {label}.\n\nXML content:\n{document_body}"
    elif document_kind == "json":
        user_msg += f"Please process the structured JSON from the paper parser using option {task}: {label}.\n\nDocument JSON:\n{document_body}"
    else:
        user_msg += f"Please process the Markdown from the paper parser using option {task}: {label}.\n\nDocument Markdown:\n{document_body}"
    if task == 2:
        instr = TASK2_DEPTH_INSTRUCTIONS.get(effective_depth, TASK2_DEPTH_INSTRUCTIONS["detailed"])
        user_msg += f"\n\n---\n\n{instr}"

    full_text = chat_complete(
        [
            {"role": "system", "content": PAPER_REVIEW_SYSTEM},
            {"role": "user", "content": user_msg},
        ],
        model=chosen_model,
        stream=True,
        print_stream=print_stream,
    )

    if not full_text.strip():
        raise RuntimeError("Model returned no text. Try another model or check rate limits.")

    upsert_review(article_id, task, chosen_model, full_text, effective_depth)
    return {
        "cached": False,
        "result": full_text,
        "model": chosen_model,
        "review_depth": effective_depth,
    }


print("Use: review_article('<article_id>', task=2, depth='detailed')")

## 9. Multi-paper context builder

Port of `app/lib/chatArticleContext.ts`. Builds a single system-prompt blob containing:

- a citation index (`Paper [1] → Internal ID … → Title`)
- the cite-link contract (`[n](cite:DOCUMENT_INTERNAL_ID)`)
- per-paper bodies: authors / year / venue / abstract / TEI plain text — totalling ≤ ~100k chars

For Related-Works modes it pulls the focused TEI Related-Work + Conclusion excerpts before the broader plain text.

In [ ]:
_MAX_TOTAL_CHARS = 100_000
_RESERVED_CHARS = 14_000
_MIN_PER_ARTICLE = 1_500


def _short_title(title: str, max_len: int = 72) -> str:
    t = (title or "").strip()
    return t if len(t) <= max_len else t[: max_len - 1] + "…"


def _build_document_body(article: dict[str, Any], focused_related_work: bool) -> str:
    chunks: list[str] = []
    if (article.get("authors") or "").strip():
        chunks.append(f"Authors:\n{article['authors'].strip()}")
    if article.get("year") is not None:
        chunks.append(f"Year: {article['year']}")
    if article.get("venue_type") or article.get("venue_name"):
        venue = " — ".join(x for x in [article.get("venue_type"), article.get("venue_name")] if x)
        chunks.append(f"Venue: {venue}")
    if (article.get("abstract") or "").strip():
        chunks.append(f"Abstract:\n{article['abstract'].strip()}")

    xml = (article.get("xml") or "").strip()
    if not xml:
        latest = get_latest_parse_output(article["id"])
        if latest and (latest.get("normalized_text") or "").strip():
            chunks.append(
                f"Extracted full text (parser={latest['parser_engine']}):\n{latest['normalized_text']}"
            )
        elif not chunks:
            return "(No extracted text for this article yet — parse the PDF first.)"
        return "\n\n".join(chunks)

    if focused_related_work:
        rel = extract_related_work_from_tei(xml)
        concl = extract_conclusion_from_tei(xml)
        if rel:
            chunks.append(f"Related work (TEI excerpt):\n{rel}")
        if concl:
            chunks.append(f"Conclusion (TEI excerpt):\n{concl}")
        chunks.append(
            "Broader context — full document (TEI → plain, may overlap excerpts above):\n"
            + strip_tei_tags(xml)
        )
    else:
        chunks.append(f"Extracted full text (TEI → plain):\n{strip_tei_tags(xml)}")

    return "\n\n".join(chunks)


def build_article_context_system_block(
    article_ids: Iterable[str],
    *,
    mode: str | None = None,  # None | "related_work_compile" | "related_work_structured"
) -> str | None:
    seen: set[str] = set()
    ids: list[str] = []
    for raw in article_ids:
        s = (raw or "").strip()
        if s and s not in seen:
            seen.add(s)
            ids.append(s)
    if not ids:
        return None

    focused = mode in ("related_work_compile", "related_work_structured")

    blobs: list[dict[str, str]] = []
    for aid in ids:
        a = get_article(aid)
        if not a:
            continue
        title = a.get("title") or a.get("pdf_path") or aid
        blobs.append({"id": aid, "title": title, "body": _build_document_body(a, focused)})
    if not blobs:
        return None

    available = max(0, _MAX_TOTAL_CHARS - _RESERVED_CHARS)
    per_article = max(_MIN_PER_ARTICLE, available // len(blobs))

    preamble = (
        "The user is asking about ONLY the following research article(s). Ground factual claims "
        "in this text only. If something is not stated here, say it is not in the provided "
        "document(s). Do not invent citations, numbers, or results."
    )
    if mode == "related_work_compile":
        preamble += (
            " Task: produce a publication-quality Related Works synthesis across these papers — "
            "thematic comparison first; avoid disconnected per-paper abstracts."
        )
    elif mode == "related_work_structured":
        preamble += (
            " Task: produce the structured related-works output requested in your system "
            "instructions (per-paper cards, synthesis, citations)."
        )

    index_map = "### Citation index (paper number → Internal ID)\n" + "\n".join(
        f"- Paper [{i + 1}] → Internal ID `{b['id']}` → {_short_title(b['title'])}"
        for i, b in enumerate(blobs)
    )

    citation_style = textwrap.dedent("""
        ### Inline citations (required for claims from the papers)
        Whenever you state a specific fact, number, method, quote, or finding from the documents, add a small superscript-style Markdown link immediately after that claim so the UI can hyperlink it:
        - Format: `[n](cite:DOCUMENT_INTERNAL_ID)` where `n` is the citation index (1, 2, 3, …) matching **Paper [n]** in the index above, and `DOCUMENT_INTERNAL_ID` is the **exact** value after `Internal ID:` in the document block (copy-paste it; never guess or invent an ID).
        - Example: `The model reached 94% accuracy[1](cite:0f1b16afa320e4c1af0fffdc79e79a11)`.
        - Use a new index `n` when pointing to a different passage or document; reuse the same `n` only when repeating the same target.
    """).strip()

    parts = [preamble, index_map, citation_style]
    for b in blobs:
        parts.append(
            f"### Document\n- Internal ID: {b['id']}\n- Title: {b['title']}\n\n{b['body'][:per_article]}"
        )
    return "\n\n---\n\n".join(parts)


print("Context builder ready.")

## 10. Multi-paper chat & lit-review synthesis

Three call patterns matching the webapp:

- `chat_over_articles(...)` — free-form question over selected papers (uses literature-synthesis system prompt).
- `compile_related_works(...)` — produces a thematic Related Works section.
- `structured_related_works(...)` — per-paper cards + cross-paper synthesis.

All three use the citation index and `[n](cite:INTERNAL_ID)` style.

In [ ]:
def _multi_paper_call(
    article_ids: list[str],
    user_msg: str,
    *,
    system_prompt: str,
    mode: str | None,
    model: str | None,
    history: list[dict[str, str]] | None,
    print_stream: bool,
) -> str:
    ctx = build_article_context_system_block(article_ids, mode=mode)
    if ctx is None:
        raise RuntimeError("No valid articles found for the provided ids.")

    messages: list[dict[str, str]] = [
        {"role": "system", "content": system_prompt},
        {"role": "system", "content": ctx},
    ]
    if history:
        messages.extend(history)
    messages.append({"role": "user", "content": user_msg})

    return chat_complete(messages, model=model, stream=True, print_stream=print_stream)


def chat_over_articles(
    article_ids: list[str],
    question: str,
    *,
    detail_level: int = 1,
    model: str | None = None,
    history: list[dict[str, str]] | None = None,
    print_stream: bool = True,
) -> str:
    """Ask a question grounded in the selected papers (lit-review synthesis style)."""
    return _multi_paper_call(
        article_ids,
        question,
        system_prompt=get_literature_synthesis_system(detail_level),
        mode=None,
        model=model,
        history=history,
        print_stream=print_stream,
    )


def compile_related_works(
    article_ids: list[str],
    *,
    detail_level: int = 1,
    extra_instructions: str | None = None,
    model: str | None = None,
    print_stream: bool = True,
) -> str:
    """Produce a thematic Related Works section across the selected papers."""
    user_msg = (
        extra_instructions
        or "Produce the Related Works section per the structure in your system instructions."
    )
    return _multi_paper_call(
        article_ids,
        user_msg,
        system_prompt=get_related_work_compile_system(detail_level),
        mode="related_work_compile",
        model=model,
        history=None,
        print_stream=print_stream,
    )


def structured_related_works(
    article_ids: list[str],
    *,
    detail_level: int = 1,
    model: str | None = None,
    print_stream: bool = True,
) -> str:
    """Per-paper cards + synthesis report."""
    return _multi_paper_call(
        article_ids,
        "Produce the structured related-works report per your system instructions.",
        system_prompt=get_related_work_structured_system(detail_level),
        mode="related_work_structured",
        model=model,
        history=None,
        print_stream=print_stream,
    )


print("Multi-paper chat ready.")

## 11. Library helpers

Quick browsing utilities mirroring `app/routes/articles.ts`.

In [ ]:
def show_library(*, search: str | None = None, limit: int = 50) -> None:
    rows = get_articles(search=search, include_xml=False)[:limit]
    print(f"{len(rows)} article(s)")
    print("-" * 80)
    for r in rows:
        title = (r.get("title") or "(untitled)")[:80]
        year = r.get("year") or "----"
        print(f"{r['id'][:10]}  {year}  {title}")
        if r.get("authors"):
            print(f"            {r['authors'][:80]}")
        if r.get("pdf_path"):
            print(f"            {r['pdf_path']}")
        print()


def show_reviews(article_id: str) -> None:
    reviews = get_reviews(article_id)
    if not reviews:
        print("No reviews yet.")
        return
    for r in reviews:
        print(f"[task {r['task']} | {r['model']} | depth={r['review_depth'] or '-'}]"
              f"  {r['created_at']}")
        print(r["result"][:500] + ("…" if len(r["result"]) > 500 else ""))
        print("-" * 80)


print(f"{article_count()} article(s), {review_count()} review(s) in DB")

## 12. Example workflow

Uncomment and run any of the following to use the pipeline. The first cell ingests one PDF, the rest demonstrate the review tasks and multi-paper chat.

In [ ]:
# --- 1) Ingest a single PDF ---
# summary = ingest_pdf("/path/to/paper.pdf")           # auto chooses best parser
# summary = ingest_pdf("/path/to/paper.pdf", prefer="grobid")
# summary = ingest_pdf("/path/to/paper.pdf", prefer="pypdf")
# print(summary)

# --- 2) Or ingest a whole folder ---
# results = ingest_directory("~/Desktop/Articles", prefer="opendataloader")

# --- 3) Browse what's in the DB ---
# show_library()
# show_library(search="diffusion")

# --- 4) Run reviews on one article ---
# aid = "<paste id from show_library>"
# review_article(aid, task=1)                          # metadata + links extraction
# review_article(aid, task=2, depth="five_line")       # section summaries
# review_article(aid, task=2, depth="detailed")
# review_article(aid, task=3)                          # related-work synthesis

# --- 5) Multi-paper chat / lit review ---
# ids = [a["id"] for a in get_articles(search="retrieval", include_xml=False)[:6]]
# chat_over_articles(ids, "What are the dominant evaluation datasets and where do results disagree?")
# compile_related_works(ids, detail_level=2)
# structured_related_works(ids, detail_level=1)

print("Notebook ready.")